In [ ]:
import pysam
import pandas as pd
import numpy as np
from Bio.Seq import Seq
from pathlib import Path
import seaborn as sns
import scipy
import re
import sklearn

import matplotlib.pyplot as plt

## Riboflow bams

In [ ]:
genome = pysam.Fastafile("/path/to/appris_human_v2_selected.fa")

gene_table = pd.read_excel('gene_lists/S17monosome_ALL_TL-DN_pvalueandTE.xlsx')

gene_table.head()

In [ ]:
def parse_riboflow_genome(genome):
    result = {}
    for reference in genome.references:
        tokens = reference.split('|')
        gene_name = tokens[5]
        cds = re.search('CDS:([^|]+)', reference).group(1)
        start, end = cds.split('-')
        start, end = int(start)-1, int(end)
        result[gene_name] = (reference, start, end)
    return result

genename_to_reference = parse_riboflow_genome(genome)

In [ ]:
codon_dfs = []
for sample in Path('/path/to/bams/').glob('*.bam'):
    samfile = pysam.AlignmentFile(sample, "rb")
    
    for gene in gene_table['gene_name']:

        if gene not in genename_to_reference:
            print(f'{gene} not in transcriptome')
            continue
        ref_name, start, end = genename_to_reference[gene]
        sequence = Seq(genome.fetch(ref_name))
        
        cds_df = pd.DataFrame(dict(
                position=range(len(sequence)),
                sequence=sequence,
                sample=sample.name,
                gene=gene,
            ))

        pileup = pd.DataFrame([(x.pos, x.n) for x in samfile.pileup(ref_name)])
        if pileup.empty:
            cds_df['counts'] = 0
        else:
            cds_df = cds_df.merge(pileup, left_on='position', right_on=0, how='left').drop(columns=[0]).rename(columns={1: 'counts'})

        cds_df = cds_df.iloc[start:end].reset_index(drop=True)
        
        # group every 3 rows into codons
        cds_df['codon'] = cds_df.index // 3
        codon_df = cds_df.groupby('codon').agg(dict(
                position='min',
                sequence='sum',
                sample='min',
                counts='sum',
                gene='min',
            )).reset_index()
        codon_df['normalized_counts'] = codon_df['counts'] / codon_df['counts'].max()  # or .sum()
        codon_dfs.append(codon_df)

codon_df = pd.concat(codon_dfs, ignore_index=True)
codon_df['time'] = pd.to_numeric(codon_df['sample'].str.extract(r'T(0|24|72)')[0])
codon_df

## Post processing

In [ ]:
def rolling_average(df, window_size, win_type='triang'):
    result = []
    for (sample, gene), sub_df in df.groupby(['sample', 'gene']):
        sub_df['rolling_counts'] = sub_df['counts'].rolling(window_size, min_periods=1, center=True, win_type=win_type).mean()
        sub_df['normalized_counts'] = sub_df['rolling_counts'] / sub_df['rolling_counts'].max()
        sub_df['sample'] = sample
        sub_df['gene'] = gene
        result.append(sub_df)
    return pd.concat(result, ignore_index=True)
        
rolling_df = rolling_average(codon_df.copy(), 11)

In [ ]:
from sklearn.metrics import r2_score

def logistic_function(x, k, x0, b):
    return (1 - b) / (1 + np.exp(-k * (x - x0))) + b

def logistic_gof(df):
    df = df.dropna()

    initial_guess = [0.5, df['codon'].mean(), df['relative_counts'].iloc[-50:].mean()]
    try:
        params, covariance = scipy.optimize.curve_fit(
            logistic_function, df['codon'], df['relative_counts'],
            bounds=([-np.inf, 0, 0], [np.inf, np.inf, 1]),
            p0=initial_guess, maxfev=5000)
    except:
        # raise
        return -1, [-1, -1, -1]
    
    y_pred = logistic_function(df['codon'], *params)
    r_squared = r2_score(df['relative_counts'], y_pred)
    
    return r_squared, params

def invert_log(V, params):
    k, x0, b = np.vstack(result['log_params'].to_numpy()).T
    return -np.log((1-b)/(V-b) -1)/k + x0
    
def extract_upstream(gene, codon, length):
    if np.isnan(codon) or codon < 0:
        return 'X' * length
    codon = int(codon)
    ref_name, start, end = genename_to_reference[gene]
    sequence = Seq(genome.fetch(ref_name))
    sequence = sequence[start:end].translate()
    start = max(codon-length, 0)
    sequence = str(sequence[start:codon])
    return 'X' * (length -len(sequence)) + sequence

def get_sequence(df):
    seqs = []
    for _, row in df.iterrows():
        seqs.append(extract_upstream(*row, 50))
    return seqs

def calculate_ratio(df, std):
    to_drop = ['sample', 'counts', 'normalized_counts', 'rolling_counts'] # for removing duplicate entries
    result = df.drop(columns=['sample', 'counts', 'normalized_counts', 'rolling_counts']).drop_duplicates().reset_index(drop=True)
    df_stats = df.groupby(['gene', 'codon'])['normalized_counts'].agg(['mean', 'std'])
    std_stats = std.groupby(['gene', 'codon'])['normalized_counts'].agg(['mean', 'std'])
    result = result.merge(df_stats['mean'].reset_index(), on=['gene', 'codon'], how='left').rename(columns={'mean': 'normalized_counts'})
    result = result.merge(std_stats['mean'].reset_index(), on=['gene', 'codon'], how='left').rename(columns={'mean': 'std_normalized_counts'})
    result = result.merge((df_stats['mean'] / std_stats['mean']).reset_index(), on=['gene', 'codon'], how='left').rename(columns={'mean': 'relative_counts'})
    result = result.merge((
        np.abs(df_stats['mean'] / std_stats['mean']) * np.sqrt(
        (df_stats['std'] / df_stats['mean']) ** 2 +
        (std_stats['std'] / std_stats['mean']) ** 2
    )
    ).reset_index(), on=['gene', 'codon'], how='left').rename(columns={0: 'relative_counts_std'})
    
    df_stats = df.groupby(['gene', 'codon'])['rolling_counts'].agg(['mean', 'std'])
    std_stats = std.groupby(['gene', 'codon'])['rolling_counts'].agg(['mean', 'std'])
    result = result.merge(df_stats['mean'].reset_index(), on=['gene', 'codon'], how='left').rename(columns={'mean': 'rolling_counts'})
    result = result.merge(std_stats['mean'].reset_index(), on=['gene', 'codon'], how='left').rename(columns={'mean': 'std_rolling_counts'})
    return result

In [ ]:
# extract some metrics of fall off
import warnings

ratio = calculate_ratio(rolling_df[rolling_df.time == 72], rolling_df[rolling_df.time == 0])

result = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    warnings.simplefilter("ignore", scipy.optimize.OptimizeWarning)
    
    for gene, dat in ratio.groupby('gene'):
        dat = dat.replace([np.inf, -np.inf], np.nan)
        # exp_gof, exp_params = exponential_gof(dat)
        log_gof, log_params = logistic_gof(dat)
        result.append({
            'gene': gene,
            'mean_counts_t0': dat['std_rolling_counts'].mean(),
            'mean_counts_t72': dat['rolling_counts'].mean(),
            'mean_relative_counts_first_50': dat[dat.codon < 50]['relative_counts'].mean(),
            'mean_relative_counts_last_150': dat[dat.codon > dat.codon.max() - 150]['relative_counts'].mean(),
            'std_relative_counts_last_150': dat[dat.codon > dat.codon.max() - 150]['relative_counts'].std(),
            # 'exponential_gof': exp_gof,
            # 'exponential_params': exp_params,
            'logistic_gof': log_gof,
            'log_params': log_params,
        })
result = pd.DataFrame(result)
result['mean_dropoff'] = result['mean_relative_counts_last_150'] / result['mean_relative_counts_first_50']
result['codon_90'] = invert_log(0.9, result['log_params'])
result['codon_50'] = invert_log(0.5, result['log_params'])
result['seq_90'] = get_sequence(result[['gene', 'codon_90']])
result['seq_50'] = get_sequence(result[['gene', 'codon_50']])
result.head()

In [ ]:
result.loc[result['logistic_gof'] == -1, 'seq_90'] = 'X' * 50
result.loc[result['logistic_gof'] == -1, 'seq_50'] = 'X' * 50
result.to_csv('Protac_falloff_metrics_w_seq.csv')

In [ ]:
result[result.logistic_gof != -1].sort_values('logistic_gof')

In [ ]:
# show logistic fit of a given gene
def show_logistic(gene):
    dat = ratio[ratio.gene == gene]
    ax = sns.lineplot(dat, x='codon', y='relative_counts', label=gene)
    xvals = np.arange(dat.codon.min(), dat.codon.max())
    yvals = logistic_function(xvals, *result.loc[result.gene == gene, 'log_params'].iloc[0])
    ax.plot(xvals, yvals, '--', color=ax.lines[-1].get_color())

show_logistic('RPS29')
show_logistic('ATP5ME')
show_logistic('PIN1')